## Multi Input CNN Architecture

#### In this setup, your RTX 3050 will process four parallel branches. Each branch extracts features from one stem, and then they are all "glued" together into a single brain.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_multi_input_model(input_shape=(128, 128, 1)):
    # 1. Define the 4 Inputs
    # What it does: This creates four distinct "tunnels" into the model.
    # Why it matters: Standard models only have one Input. Here, we tell TensorFlow to expect four separate Mel-Spectrograms
    #  (each 128×128) for every single training example. It ensures the Vocal data never touches the Drum data at the start.


    in_vocal = layers.Input(shape=input_shape, name="vocal_in")
    in_other = layers.Input(shape=input_shape, name="other_in")
    in_drum  = layers.Input(shape=input_shape, name="drum_in")
    in_bass  = layers.Input(shape=input_shape, name="bass_in")

    # 2. Shared Feature Extractor (Mini-CNN)
    # We use a function to keep the code clean
    def extract_features(x):
        x = layers.Conv2D(32, (3, 3), activation='relu')(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Conv2D(64, (3, 3), activation='relu')(x)
        x = layers.GlobalAveragePooling2D()(x)
        return x

    # 3. Process all 4 stems
    v_feat = extract_features(in_vocal)
    o_feat = extract_features(in_other)
    d_feat = extract_features(in_drum)
    b_feat = extract_features(in_bass)

    # 4. CONCATENATION (The Relationship Layer)
    # This is where the model learns the "Relationship"
    merged = layers.Concatenate()([v_feat, o_feat, d_feat, b_feat])

    # 5. Final Classification
    x = layers.Dense(128, activation='relu')(merged)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(2, activation='softmax')(x) # 2 for Bollypop vs Semiclassical

    model = models.Model(inputs=[in_vocal, in_other, in_drum, in_bass], outputs=output)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    return model

# Create the model
multi_model = build_multi_input_model()